#**SuFinex Synthetic Data Generator**

This notebook generates realistic synthetic financial data for the SuFinex Financial Intelligence Platform.

## **Objectives**

- Generate synthetic financial entities
- Maintain database relationships
- Simulate realistic customer behavior
- Generate transaction activity
- Simulate fraud, risk, and churn signals
- Load generated data into PostgreSQL

We're installing:

- Faker -> realistic names, cities, companies, etc.
- Pandas -> data manipulation
- NumPy -> numerical/random generation
- psycopg2-binary -> PostgreSQL connection
- SQLAlchemy -> database interaction  ->

In [84]:
!pip install faker pandas numpy psycopg2-binary sqlalchemy -q

In [85]:
import numpy as np
import pandas as pd
from faker import Faker
import random
from datetime import datetime, timedelta
import psycopg2
from sqlalchemy import create_engine, text

fake = Faker("en_IN")

# Reproducibility
np.random.seed(42)
Faker.seed(42)

print("SuFinex Synthetic Data Generator initialized successfully.")

SuFinex Synthetic Data Generator initialized successfully.


## Project Configuration

This section defines the scale, time period, and basic parameters used to generate the SuFinex synthetic dataset.

In [86]:
# ============================================================
# SuFinex Synthetic Data Configuration
# ============================================================

NUM_INSTITUTIONS = 5
NUM_CUSTOMERS = 10000
NUM_ACCOUNTS = 15000
NUM_CARDS = 12000
NUM_CREDIT_PROFILES = 10000
NUM_LOANS = 4000
NUM_MERCHANTS = 2000
NUM_PAYMENT_METHODS = 6
NUM_DEVICES = 12000
NUM_LOCATIONS = 500
NUM_SUPPORT_TICKETS = 5000
NUM_TRANSACTIONS = 100000

START_DATE = "2024-01-01"
END_DATE = "2025-12-31"

CURRENCY = "INR"

print("Configuration loaded successfully.")

Configuration loaded successfully.


In [87]:
# ============================================================
# ID Generation Helpers
# ============================================================


def generate_ids(prefix, count, width=6):
    """
    Generate unique IDs such as CUS000001, CUS000002, etc.
    """

    ids = []

    for i in range(1, count + 1):
        id_value = f"{prefix}{i:0{width}d}"
        ids.append(id_value)

    return ids


customer_ids = generate_ids("CUS", 5)

print(customer_ids)

['CUS000001', 'CUS000002', 'CUS000003', 'CUS000004', 'CUS000005']


## **1. Institution Data Generation**

Institutions represent financial organizations operating within the SuFinex platform.

In [88]:
# Number of institutions
num_institutions = NUM_INSTITUTIONS

# Synthetic institution names
institution_names = [
    "Apex Bank",
    "Nova Financial",
    "Prime Credit Union",
    "Urban Finance",
    "Vertex Payments"
]

# Institution types
institution_types = [
    "Bank",
    "Bank",
    "Credit Union",
    "NBFC",
    "Fintech"
]

# Country and status
countries = ["India"] * num_institutions
statuses = ["Active"] * num_institutions

In [89]:
# Generate the institution IDs
institution_ids = generate_ids("INS", num_institutions)

print(institution_ids)

['INS000001', 'INS000002', 'INS000003', 'INS000004', 'INS000005']


In [90]:
# Create the Institution DataFrame

institutions_df = pd.DataFrame({
    "institution_id": institution_ids,
    "institution_name": institution_names,
    "institution_type": institution_types,
    "country": countries,
    "status": statuses
})

institutions_df

,institution_id,institution_name,institution_type,country,status
0,INS000001,Apex Bank,Bank,India,Active
1,INS000002,Nova Financial,Bank,India,Active
2,INS000003,Prime Credit Union,Credit Union,India,Active
3,INS000004,Urban Finance,NBFC,India,Active
4,INS000005,Vertex Payments,Fintech,India,Active


In [91]:
# Validate the dataset

print("Number of institutions:", len(institutions_df))
print("Number of columns:", len(institutions_df.columns))

print("\nInstitution IDs:")
print(institutions_df["institution_id"].tolist())

print("\nMissing values:")
print(institutions_df.isnull().sum())

Number of institutions: 5
Number of columns: 5

Institution IDs:
['INS000001', 'INS000002', 'INS000003', 'INS000004', 'INS000005']

Missing values:
institution_id      0
institution_name    0
institution_type    0
country             0
status              0
dtype: int64


## 2. **Customer Data Generation**

Customers represent individuals associated with financial institutions.

Customer attributes will be synthetically generated and will later support customer segmentation, risk analysis, churn prediction, and transaction behavior analysis.

### **Generate Customer IDs**

In [92]:
# Number of customers
num_customers = NUM_CUSTOMERS

# Generate customer IDs
customer_ids = generate_ids("CUS", num_customers)

print("Number of customer IDs generated:", len(customer_ids))
print("First 5 IDs:", customer_ids[:5])

Number of customer IDs generated: 10000
First 5 IDs: ['CUS000001', 'CUS000002', 'CUS000003', 'CUS000004', 'CUS000005']


### **Generate Institution Relationships**

In [93]:
# Randomly assign each customer to one of our institutions
customer_institution_ids = np.random.choice(
    institutions_df["institution_id"],
    size=num_customers
)

print("First 10 institution assignments:")
print(customer_institution_ids[:10])

First 10 institution assignments:
['INS000004' 'INS000005' 'INS000003' 'INS000005' 'INS000005' 'INS000002'
 'INS000003' 'INS000003' 'INS000003' 'INS000005']


### **Generate Customer Names**

In [94]:
first_names = [fake.first_name() for _ in range(num_customers)]
last_names = [fake.last_name() for _ in range(num_customers)]

print("Sample customers:")
for i in range(5):
    print(first_names[i], last_names[i])

Sample customers:
Isaac Kota
Aryan Sodhi
Anvi Choudhary
Yash Bath
Udant Badal


### **Generate Customer Attributes**

In [95]:
date_of_birth = [
    fake.date_of_birth(minimum_age=18, maximum_age=75)
    for _ in range(num_customers)
]

genders = np.random.choice(
    ["Male", "Female", "Other"],
    size=num_customers,
    p=[0.48, 0.48, 0.04]
)

cities = [
    fake.city()
    for _ in range(num_customers)
]

states = [
    fake.state()
    for _ in range(num_customers)
]

countries = ["India"] * num_customers

customer_since = [
    fake.date_between(
        start_date="-8y",
        end_date="today"
    )
    for _ in range(num_customers)
]

customer_status = np.random.choice(
    ["Active", "Inactive"],
    size=num_customers,
    p=[0.90, 0.10]
)

### **Create the Customer DataFrame**

In [96]:
customers_df = pd.DataFrame({
    "customer_id": customer_ids,
    "institution_id": customer_institution_ids,
    "first_name": first_names,
    "last_name": last_names,
    "date_of_birth": date_of_birth,
    "gender": genders,
    "city": cities,
    "state": states,
    "country": countries,
    "customer_since": customer_since,
    "customer_status": customer_status
})

customers_df.head()

,customer_id,institution_id,first_name,last_name,date_of_birth,gender,city,state,country,customer_since,customer_status
0,CUS000001,INS000004,Isaac,Kota,2004-03-24,Male,Satna,Tripura,India,2021-04-06,Inactive
1,CUS000002,INS000005,Aryan,Sodhi,1984-07-17,Female,Ludhiana,Uttarakhand,India,2021-09-24,Active
2,CUS000003,INS000003,Anvi,Choudhary,1963-06-03,Male,Chapra,Maharashtra,India,2022-04-12,Active
3,CUS000004,INS000005,Yash,Bath,1996-05-19,Male,Tiruppur,Goa,India,2020-12-14,Active
4,CUS000005,INS000005,Udant,Badal,1955-10-15,Male,Khandwa,Odisha,India,2020-08-01,Active


### **Validate the Customers data**

In [97]:
print("Number of customers:", len(customers_df))
print("Number of columns:", len(customers_df.columns))

print("\nMissing values:")
print(customers_df.isnull().sum())

print("\nDuplicate customer IDs:",
      customers_df["customer_id"].duplicated().sum())

print("\nCustomers by institution:")
print(customers_df["institution_id"].value_counts())

Number of customers: 10000
Number of columns: 11

Missing values:
customer_id        0
institution_id     0
first_name         0
last_name          0
date_of_birth      0
gender             0
city               0
state              0
country            0
customer_since     0
customer_status    0
dtype: int64

Duplicate customer IDs: 0

Customers by institution:
institution_id
INS000001    2047
INS000005    2019
INS000002    2016
INS000004    1975
INS000003    1943
Name: count, dtype: int64


In [98]:
# Indian city and state mapping
city_state_map = {
    "Mumbai": "Maharashtra",
    "Pune": "Maharashtra",
    "Thane": "Maharashtra",
    "Nagpur": "Maharashtra",
    "Bengaluru": "Karnataka",
    "Mysuru": "Karnataka",
    "Hyderabad": "Telangana",
    "Chennai": "Tamil Nadu",
    "Ahmedabad": "Gujarat",
    "Surat": "Gujarat",
    "Delhi": "Delhi",
    "Jaipur": "Rajasthan",
    "Kolkata": "West Bengal",
    "Kochi": "Kerala",
    "Lucknow": "Uttar Pradesh",
    "Indore": "Madhya Pradesh",
    "Bhopal": "Madhya Pradesh",
    "Patna": "Bihar",
    "Bhubaneswar": "Odisha",
    "Chandigarh": "Chandigarh"
}

cities_list = list(city_state_map.keys())

print("Number of cities:", len(cities_list))

Number of cities: 20


In [99]:
customer_cities = np.random.choice(
    cities_list,
    size=num_customers
)

customer_states = [
    city_state_map[city]
    for city in customer_cities
]

print("Sample locations:")

for i in range(5):
    print(customer_cities[i], "-->", customer_states[i])

Sample locations:
Patna --> Bihar
Thane --> Maharashtra
Mumbai --> Maharashtra
Mumbai --> Maharashtra
Kochi --> Kerala


In [100]:
customers_df["city"] = customer_cities
customers_df["state"] = customer_states

customers_df.head()

,customer_id,institution_id,first_name,last_name,date_of_birth,gender,city,state,country,customer_since,customer_status
0,CUS000001,INS000004,Isaac,Kota,2004-03-24,Male,Patna,Bihar,India,2021-04-06,Inactive
1,CUS000002,INS000005,Aryan,Sodhi,1984-07-17,Female,Thane,Maharashtra,India,2021-09-24,Active
2,CUS000003,INS000003,Anvi,Choudhary,1963-06-03,Male,Mumbai,Maharashtra,India,2022-04-12,Active
3,CUS000004,INS000005,Yash,Bath,1996-05-19,Male,Mumbai,Maharashtra,India,2020-12-14,Active
4,CUS000005,INS000005,Udant,Badal,1955-10-15,Male,Kochi,Kerala,India,2020-08-01,Active


In [101]:
customers_df.tail()

,customer_id,institution_id,first_name,last_name,date_of_birth,gender,city,state,country,customer_since,customer_status
9995,CUS009996,INS000002,Aditya,Batta,1989-01-02,Female,Mysuru,Karnataka,India,2025-12-20,Active
9996,CUS009997,INS000003,Indira,Prabhakar,2006-04-06,Female,Bengaluru,Karnataka,India,2023-11-26,Active
9997,CUS009998,INS000002,Urvi,Mutti,1979-10-28,Male,Bhubaneswar,Odisha,India,2022-08-05,Active
9998,CUS009999,INS000004,Aarav,Divan,1955-01-21,Female,Lucknow,Uttar Pradesh,India,2021-09-04,Active
9999,CUS010000,INS000003,Vrinda,Sandal,1988-06-08,Male,Surat,Gujarat,India,2022-06-30,Active


## 3. **Accounts Data Generation**

Accounts represent customer banking relationships, balances, and account lifecycle details.

In [102]:
def generate_accounts(customers_df, institutions_df):
    """
    Generate synthetic customer bank accounts.
    """

    account_ids = generate_ids("ACC", NUM_ACCOUNTS)

    # Assign each account to an existing customer
    account_customer_ids = np.random.choice(
        customers_df["customer_id"],
        size=NUM_ACCOUNTS
    )

    # Get the institution associated with each customer
    customer_institution_map = dict(
        zip(
            customers_df["customer_id"],
            customers_df["institution_id"]
        )
    )

    account_institution_ids = [
        customer_institution_map[customer_id]
        for customer_id in account_customer_ids
    ]

    account_types = np.random.choice(
        ["Savings", "Current", "Salary"],
        size=NUM_ACCOUNTS,
        p=[0.65, 0.20, 0.15]
    )

    account_numbers = [
        fake.numerify(text="################")
        for _ in range(NUM_ACCOUNTS)
    ]

    balances = np.round(
        np.random.lognormal(
            mean=10,
            sigma=1,
            size=NUM_ACCOUNTS
        ),
        2
    )

    opened_dates = [
        fake.date_between(
            start_date="-8y",
            end_date="today"
        )
        for _ in range(NUM_ACCOUNTS)
    ]

    account_statuses = np.random.choice(
        ["Active", "Inactive", "Closed"],
        size=NUM_ACCOUNTS,
        p=[0.85, 0.10, 0.05]
    )

    accounts_df = pd.DataFrame({
        "account_id": account_ids,
        "customer_id": account_customer_ids,
        "institution_id": account_institution_ids,
        "account_type": account_types,
        "account_number": account_numbers,
        "balance": balances,
        "opened_date": opened_dates,
        "account_status": account_statuses
    })

    return accounts_df

In [103]:
accounts_df = generate_accounts(
    customers_df,
    institutions_df
)

print("Accounts generated:", len(accounts_df))

accounts_df.head()

Accounts generated: 15000


,account_id,customer_id,institution_id,account_type,account_number,balance,opened_date,account_status
0,ACC000001,CUS005991,INS000001,Savings,5136819480446358,85409.64,2019-11-17,Active
1,ACC000002,CUS001683,INS000003,Savings,0027291714112129,10335.92,2019-08-10,Active
2,ACC000003,CUS001967,INS000002,Savings,0588841936263134,21937.85,2023-01-21,Active
3,ACC000004,CUS003287,INS000001,Savings,9460781829521938,66439.15,2019-08-02,Active
4,ACC000005,CUS000996,INS000002,Savings,5222931142311547,17804.17,2023-06-02,Closed


## 4. **Cards Data Generation**

Cards store debit and credit card information linked to customer accounts.


In [104]:
def generate_cards(accounts_df, customers_df):
    """
    Generate synthetic payment cards linked to customer accounts.
    """

    card_ids = generate_ids("CARD", NUM_CARDS)

    # Assign cards to existing accounts
    card_account_ids = np.random.choice(
        accounts_df["account_id"],
        size=NUM_CARDS
    )

    # Map account → customer
    account_customer_map = dict(
        zip(
            accounts_df["account_id"],
            accounts_df["customer_id"]
        )
    )

    card_customer_ids = [
        account_customer_map[account_id]
        for account_id in card_account_ids
    ]

    card_types = np.random.choice(
        ["Debit", "Credit"],
        size=NUM_CARDS,
        p=[0.70, 0.30]
    )

    card_networks = np.random.choice(
        ["Visa", "Mastercard", "RuPay"],
        size=NUM_CARDS,
        p=[0.40, 0.35, 0.25]
    )

    card_statuses = np.random.choice(
        ["Active", "Blocked", "Expired"],
        size=NUM_CARDS,
        p=[0.90, 0.06, 0.04]
    )

    issue_dates = [
        fake.date_between(
            start_date="-5y",
            end_date="today"
        )
        for _ in range(NUM_CARDS)
    ]

    expiry_dates = [
        issue_date.replace(
            year=issue_date.year + 4
        )
        for issue_date in issue_dates
    ]

    cards_df = pd.DataFrame({
        "card_id": card_ids,
        "customer_id": card_customer_ids,
        "account_id": card_account_ids,
        "card_type": card_types,
        "card_network": card_networks,
        "issue_date": issue_dates,
        "expiry_date": expiry_dates,
        "card_status": card_statuses
    })

    return cards_df

In [105]:
cards_df = generate_cards(
    accounts_df,
    customers_df
)

print("Cards generated:", len(cards_df))

cards_df.head()

Cards generated: 12000


,card_id,customer_id,account_id,card_type,card_network,issue_date,expiry_date,card_status
0,CARD000001,CUS008932,ACC010571,Debit,Mastercard,2022-02-08,2026-02-08,Active
1,CARD000002,CUS001488,ACC004485,Debit,Mastercard,2022-07-24,2026-07-24,Active
2,CARD000003,CUS008156,ACC012898,Debit,Mastercard,2022-05-20,2026-05-20,Active
3,CARD000004,CUS005079,ACC002457,Credit,Visa,2026-03-21,2030-03-21,Active
4,CARD000005,CUS008382,ACC004481,Debit,Visa,2025-12-30,2029-12-30,Active


## 5. **Credit Profiles Data Generation**

Credit profiles capture customer creditworthiness, utilization, and repayment behavior indicators.

In [106]:
def generate_credit_profiles(customers_df):
    """
    Generate one synthetic credit profile for each customer.
    """

    credit_profile_ids = generate_ids(
        "CRP",
        len(customers_df)
    )

    customer_ids = customers_df["customer_id"].tolist()

    credit_scores = np.random.randint(
        300,
        901,
        size=len(customers_df)
    )

    credit_utilization = np.round(
        np.random.uniform(
            0,
            95,
            size=len(customers_df)
        ),
        2
    )

    total_credit_limit = np.round(
        np.random.lognormal(
            mean=10.5,
            sigma=0.8,
            size=len(customers_df)
        ),
        2
    )

    outstanding_balance = np.round(
        total_credit_limit *
        (credit_utilization / 100),
        2
    )

    delinquency_count = np.random.poisson(
        lam=0.4,
        size=len(customers_df)
    )

    last_updated = [
        fake.date_between(
            start_date="-1y",
            end_date="today"
        )
        for _ in range(len(customers_df))
    ]

    credit_profiles_df = pd.DataFrame({
        "credit_profile_id": credit_profile_ids,
        "customer_id": customer_ids,
        "credit_score": credit_scores,
        "credit_utilization": credit_utilization,
        "total_credit_limit": total_credit_limit,
        "outstanding_balance": outstanding_balance,
        "delinquency_count": delinquency_count,
        "last_updated": last_updated
    })

    return credit_profiles_df

In [107]:
credit_profiles_df = generate_credit_profiles(
    customers_df
)

print(
    "Credit profiles generated:",
    len(credit_profiles_df)
)

credit_profiles_df.head()

Credit profiles generated: 10000


,credit_profile_id,customer_id,credit_score,credit_utilization,total_credit_limit,outstanding_balance,delinquency_count,last_updated
0,CRP000001,CUS000001,495,76.37,41246.19,31499.72,0,2025-11-03
1,CRP000002,CUS000002,303,83.20,47347.99,39393.53,0,2026-07-05
2,CRP000003,CUS000003,478,68.45,18222.12,12473.04,1,2026-07-16
3,CRP000004,CUS000004,523,66.60,18761.01,12494.83,0,2025-11-23
4,CRP000005,CUS000005,466,21.97,9609.07,2111.11,0,2026-06-05


### **Validate Data**

In [108]:


print("\nAccounts:", len(accounts_df))
print("Cards:", len(cards_df))
print("Credit Profiles:", len(credit_profiles_df))

print("\nDuplicate Account IDs:",
      accounts_df["account_id"].duplicated().sum())

print("Duplicate Card IDs:",
      cards_df["card_id"].duplicated().sum())

print("Duplicate Credit Profile IDs:",
      credit_profiles_df["credit_profile_id"].duplicated().sum())

print("\nCredit Score Range:")
print(
    credit_profiles_df["credit_score"].min(),
    "to",
    credit_profiles_df["credit_score"].max()
)

print("\nMissing Values:")
print("\nAccounts:")
print(accounts_df.isnull().sum())

print("\nCards:")
print(cards_df.isnull().sum())

print("\nCredit Profiles:")
print(credit_profiles_df.isnull().sum())


Accounts: 15000
Cards: 12000
Credit Profiles: 10000

Duplicate Account IDs: 0
Duplicate Card IDs: 0
Duplicate Credit Profile IDs: 0

Credit Score Range:
300 to 900

Missing Values:

Accounts:
account_id        0
customer_id       0
institution_id    0
account_type      0
account_number    0
balance           0
opened_date       0
account_status    0
dtype: int64

Cards:
card_id         0
customer_id     0
account_id      0
card_type       0
card_network    0
issue_date      0
expiry_date     0
card_status     0
dtype: int64

Credit Profiles:
credit_profile_id      0
customer_id            0
credit_score           0
credit_utilization     0
total_credit_limit     0
outstanding_balance    0
delinquency_count      0
last_updated           0
dtype: int64


## 6. **Loans Data Generation**

Loans track customer borrowing activities, loan amounts, tenure, and repayment status.

In [109]:
def generate_loans(customers_df):
    """
    Generate synthetic loans linked to existing customers.
    """

    loan_ids = generate_ids("LOAN", NUM_LOANS)

    # Select customers who will have loans
    loan_customer_ids = np.random.choice(
        customers_df["customer_id"],
        size=NUM_LOANS,
        replace=False
    )

    # Map customer to institution
    customer_institution_map = dict(
        zip(
            customers_df["customer_id"],
            customers_df["institution_id"]
        )
    )

    loan_institution_ids = [
        customer_institution_map[customer_id]
        for customer_id in loan_customer_ids
    ]

    loan_types = np.random.choice(
        ["Personal", "Home", "Auto", "Education"],
        size=NUM_LOANS,
        p=[0.45, 0.20, 0.20, 0.15]
    )

    loan_amounts = np.round(
        np.random.lognormal(
            mean=12,
            sigma=0.8,
            size=NUM_LOANS
        ),
        2
    )

    interest_rates = np.round(
        np.random.uniform(
            7.5,
            18.0,
            size=NUM_LOANS
        ),
        2
    )

    loan_statuses = np.random.choice(
        ["Active", "Closed", "Defaulted"],
        size=NUM_LOANS,
        p=[0.75, 0.20, 0.05]
    )

    loan_start_dates = [
        fake.date_between(
            start_date="-5y",
            end_date="today"
        )
        for _ in range(NUM_LOANS)
    ]

    loan_tenures = np.random.choice(
        [12, 24, 36, 48, 60],
        size=NUM_LOANS
    )

    loans_df = pd.DataFrame({
        "loan_id": loan_ids,
        "customer_id": loan_customer_ids,
        "institution_id": loan_institution_ids,
        "loan_type": loan_types,
        "loan_amount": loan_amounts,
        "interest_rate": interest_rates,
        "loan_start_date": loan_start_dates,
        "loan_tenure_months": loan_tenures,
        "loan_status": loan_statuses
    })

    return loans_df

In [110]:
loans_df = generate_loans(customers_df)

print("Loans generated:", len(loans_df))

loans_df.head()

Loans generated: 4000


,loan_id,customer_id,institution_id,loan_type,loan_amount,interest_rate,loan_start_date,loan_tenure_months,loan_status
0,LOAN000001,CUS001290,INS000005,Education,80985.07,8.99,2025-11-11,12,Closed
1,LOAN000002,CUS004695,INS000002,Personal,236789.49,8.71,2024-03-06,36,Closed
2,LOAN000003,CUS008538,INS000005,Education,56111.53,11.20,2022-05-14,48,Active
3,LOAN000004,CUS006211,INS000004,Personal,116841.06,11.55,2025-09-08,48,Active
4,LOAN000005,CUS007536,INS000005,Education,90674.83,12.26,2021-12-19,12,Active


## 7. **Merchants Data Generation**

Merchants represent businesses where customers perform financial transactions.

In [111]:
def generate_merchants(institutions_df):
    """
    Generate synthetic merchants associated with institutions.
    """

    merchant_ids = generate_ids("MER", NUM_MERCHANTS)

    merchant_institution_ids = np.random.choice(
        institutions_df["institution_id"],
        size=NUM_MERCHANTS
    )

    merchant_categories = np.random.choice(
        [
            "Grocery",
            "Restaurant",
            "Fuel",
            "Travel",
            "Electronics",
            "Healthcare",
            "Shopping",
            "Entertainment"
        ],
        size=NUM_MERCHANTS
    )

    merchant_names = [
        f"{fake.company()} Store"
        for _ in range(NUM_MERCHANTS)
    ]

    cities = np.random.choice(
        cities_list,
        size=NUM_MERCHANTS
    )

    merchant_statuses = np.random.choice(
        ["Active", "Inactive"],
        size=NUM_MERCHANTS,
        p=[0.95, 0.05]
    )

    merchants_df = pd.DataFrame({
        "merchant_id": merchant_ids,
        "institution_id": merchant_institution_ids,
        "merchant_name": merchant_names,
        "merchant_category": merchant_categories,
        "city": cities,
        "merchant_status": merchant_statuses
    })

    return merchants_df

In [112]:
merchants_df = generate_merchants(institutions_df)

print("Merchants generated:", len(merchants_df))

merchants_df.head()

Merchants generated: 2000


,merchant_id,institution_id,merchant_name,merchant_category,city,merchant_status
0,MER000001,INS000001,Mann-Sheth Store,Healthcare,Chennai,Active
1,MER000002,INS000002,"Dave, Suri and Nagarajan Store",Electronics,Chennai,Active
2,MER000003,INS000005,Chopra PLC Store,Electronics,Delhi,Active
3,MER000004,INS000002,Bail-Chowdhury Store,Grocery,Chennai,Active
4,MER000005,INS000002,"Savant, Pant and Iyer Store",Restaurant,Mysuru,Active


## 8. **Payment Methods Data Generation**

Payment methods define the available transaction channels used across the platform.

In [113]:
def generate_payment_methods():
    """
    Generate synthetic payment method reference data.
    """

    payment_method_ids = generate_ids(
        "PM",
        NUM_PAYMENT_METHODS
    )

    payment_methods = [
        "UPI",
        "Debit Card",
        "Credit Card",
        "Net Banking",
        "Wallet",
        "Bank Transfer"
    ]

    payment_method_status = [
        "Active"
    ] * NUM_PAYMENT_METHODS

    payment_methods_df = pd.DataFrame({
        "payment_method_id": payment_method_ids,
        "payment_method_name": payment_methods,
        "status": payment_method_status
    })

    return payment_methods_df

In [114]:
payment_methods_df = generate_payment_methods()

print("Payment methods generated:", len(payment_methods_df))

payment_methods_df

Payment methods generated: 6


,payment_method_id,payment_method_name,status
0,PM000001,UPI,Active
1,PM000002,Debit Card,Active
2,PM000003,Credit Card,Active
3,PM000004,Net Banking,Active
4,PM000005,Wallet,Active
5,PM000006,Bank Transfer,Active



## 9. **Devices Data Generation**

Devices capture customer device information used to access financial services.

In [115]:
def generate_devices(customers_df):
    """
    Generate synthetic devices linked to customers.
    """

    device_ids = generate_ids("DEV", NUM_DEVICES)

    device_customer_ids = np.random.choice(
        customers_df["customer_id"],
        size=NUM_DEVICES
    )

    device_types = np.random.choice(
        ["Mobile", "Laptop", "Tablet"],
        size=NUM_DEVICES,
        p=[0.70, 0.20, 0.10]
    )

    operating_systems = np.random.choice(
        ["Android", "iOS", "Windows", "macOS"],
        size=NUM_DEVICES,
        p=[0.50, 0.25, 0.15, 0.10]
    )

    device_statuses = np.random.choice(
        ["Active", "Inactive"],
        size=NUM_DEVICES,
        p=[0.90, 0.10]
    )

    devices_df = pd.DataFrame({
        "device_id": device_ids,
        "customer_id": device_customer_ids,
        "device_type": device_types,
        "operating_system": operating_systems,
        "device_status": device_statuses
    })

    return devices_df

In [116]:
devices_df = generate_devices(customers_df)

print("Devices generated:", len(devices_df))

devices_df.head()

Devices generated: 12000


,device_id,customer_id,device_type,operating_system,device_status
0,DEV000001,CUS009338,Mobile,iOS,Active
1,DEV000002,CUS007473,Mobile,Android,Inactive
2,DEV000003,CUS003830,Laptop,Android,Active
3,DEV000004,CUS006597,Mobile,iOS,Active
4,DEV000005,CUS006559,Mobile,Windows,Active



## 10: **Locations Data Generation**

Locations provide geographical reference data used for customer and transaction analytics.


In [117]:
def generate_locations():
    """
    Generate synthetic Indian transaction locations.
    """

    location_ids = generate_ids(
        "LOC",
        NUM_LOCATIONS
    )

    location_cities = np.random.choice(
        cities_list,
        size=NUM_LOCATIONS
    )

    location_states = [
        city_state_map[city]
        for city in location_cities
    ]

    locations_df = pd.DataFrame({
        "location_id": location_ids,
        "city": location_cities,
        "state": location_states,
        "country": ["India"] * NUM_LOCATIONS
    })

    return locations_df

In [118]:
locations_df = generate_locations()

print("Locations generated:", len(locations_df))

locations_df.head()

Locations generated: 500


,location_id,city,state,country
0,LOC000001,Kolkata,West Bengal,India
1,LOC000002,Surat,Gujarat,India
2,LOC000003,Mumbai,Maharashtra,India
3,LOC000004,Mysuru,Karnataka,India
4,LOC000005,Pune,Maharashtra,India


### **Validate Data**

In [119]:

print("\nLoans:", len(loans_df))
print("Merchants:", len(merchants_df))
print("Payment Methods:", len(payment_methods_df))
print("Devices:", len(devices_df))
print("Locations:", len(locations_df))

print("\nDuplicate IDs:")

print(
    "Loan IDs:",
    loans_df["loan_id"].duplicated().sum()
)

print(
    "Merchant IDs:",
    merchants_df["merchant_id"].duplicated().sum()
)

print(
    "Payment Method IDs:",
    payment_methods_df["payment_method_id"].duplicated().sum()
)

print(
    "Device IDs:",
    devices_df["device_id"].duplicated().sum()
)

print(
    "Location IDs:",
    locations_df["location_id"].duplicated().sum()
)


Loans: 4000
Merchants: 2000
Payment Methods: 6
Devices: 12000
Locations: 500

Duplicate IDs:
Loan IDs: 0
Merchant IDs: 0
Payment Method IDs: 0
Device IDs: 0
Location IDs: 0


## 11:**Support Tickets Data Generation**

Support tickets record customer service interactions, complaints, and issue resolution workflows.

In [120]:
def generate_support_tickets(customers_df):
    """
    Generate synthetic customer support tickets.
    """

    ticket_ids = generate_ids(
        "TKT",
        NUM_SUPPORT_TICKETS
    )

    # Each ticket belongs to an existing customer
    ticket_customer_ids = np.random.choice(
        customers_df["customer_id"],
        size=NUM_SUPPORT_TICKETS
    )

    # Map customer to institution
    customer_institution_map = dict(
        zip(
            customers_df["customer_id"],
            customers_df["institution_id"]
        )
    )

    ticket_institution_ids = [
        customer_institution_map[customer_id]
        for customer_id in ticket_customer_ids
    ]

    ticket_categories = np.random.choice(
        [
            "Transaction Issue",
            "Card Issue",
            "Account Issue",
            "Loan Query",
            "Fraud Report",
            "Payment Issue",
            "Technical Issue"
        ],
        size=NUM_SUPPORT_TICKETS,
        p=[0.20, 0.15, 0.15, 0.10, 0.10, 0.20, 0.10]
    )

    ticket_priorities = np.random.choice(
        ["Low", "Medium", "High", "Critical"],
        size=NUM_SUPPORT_TICKETS,
        p=[0.35, 0.40, 0.20, 0.05]
    )

    ticket_statuses = np.random.choice(
        ["Open", "In Progress", "Resolved", "Closed"],
        size=NUM_SUPPORT_TICKETS,
        p=[0.10, 0.10, 0.50, 0.30]
    )

    ticket_channels = np.random.choice(
        ["Mobile App", "Website", "Phone", "Email", "Branch"],
        size=NUM_SUPPORT_TICKETS,
        p=[0.35, 0.20, 0.20, 0.15, 0.10]
    )

    created_dates = [
        fake.date_between(
            start_date="-2y",
            end_date="today"
        )
        for _ in range(NUM_SUPPORT_TICKETS)
    ]

    resolution_days = np.random.randint(
        1,
        8,
        size=NUM_SUPPORT_TICKETS
    )

    support_tickets_df = pd.DataFrame({
        "ticket_id": ticket_ids,
        "customer_id": ticket_customer_ids,
        "institution_id": ticket_institution_ids,
        "ticket_category": ticket_categories,
        "priority": ticket_priorities,
        "status": ticket_statuses,
        "channel": ticket_channels,
        "created_date": created_dates,
        "resolution_days": resolution_days
    })

    return support_tickets_df

In [121]:
support_tickets_df = generate_support_tickets(
    customers_df
)

print(
    "Support tickets generated:",
    len(support_tickets_df)
)

support_tickets_df.head()

Support tickets generated: 5000


,ticket_id,customer_id,institution_id,ticket_category,priority,status,channel,created_date,resolution_days
0,TKT000001,CUS009753,INS000001,Loan Query,High,Resolved,Email,2026-01-28,2
1,TKT000002,CUS001221,INS000002,Fraud Report,Low,Closed,Email,2025-07-13,6
2,TKT000003,CUS004502,INS000005,Account Issue,High,Closed,Email,2026-05-03,5
3,TKT000004,CUS002511,INS000002,Transaction Issue,Low,Closed,Mobile App,2026-04-26,3
4,TKT000005,CUS007785,INS000002,Payment Issue,Medium,Resolved,Mobile App,2026-03-28,1


In [122]:
print("========== SUPPORT TICKET VALIDATION ==========")

print(
    "Rows:",
    len(support_tickets_df)
)

print(
    "Duplicate Ticket IDs:",
    support_tickets_df["ticket_id"].duplicated().sum()
)

print("\nMissing Values:")
print(support_tickets_df.isnull().sum())

print("\nTicket Categories:")
print(support_tickets_df["ticket_category"].value_counts())

print("\nTicket Priorities:")
print(support_tickets_df["priority"].value_counts())

print("\nTicket Status:")
print(support_tickets_df["status"].value_counts())

========== SUPPORT TICKET VALIDATION ==========
Rows: 5000
Duplicate Ticket IDs: 0

Missing Values:
ticket_id          0
customer_id        0
institution_id     0
ticket_category    0
priority           0
status             0
channel            0
created_date       0
resolution_days    0
dtype: int64

Ticket Categories:
ticket_category
Payment Issue        1056
Transaction Issue     957
Card Issue            769
Account Issue         717
Fraud Report          530
Loan Query            489
Technical Issue       482
Name: count, dtype: int64

Ticket Priorities:
priority
Medium      1982
Low         1746
High        1019
Critical     253
Name: count, dtype: int64

Ticket Status:
status
Resolved       2473
Closed         1483
In Progress     537
Open            507
Name: count, dtype: int64


## 12:**Transactions Data Generation**

Transactions store financial activity performed by customers across accounts, merchants, and payment channels.


In [123]:
print("Accounts:")
print(accounts_df.columns.tolist())

print("\nMerchants:")
print(merchants_df.columns.tolist())

print("\nPayment Methods:")
print(payment_methods_df.columns.tolist())

print("\nDevices:")
print(devices_df.columns.tolist())

print("\nLocations:")
print(locations_df.columns.tolist())

Accounts:
['account_id', 'customer_id', 'institution_id', 'account_type', 'account_number', 'balance', 'opened_date', 'account_status']

Merchants:
['merchant_id', 'institution_id', 'merchant_name', 'merchant_category', 'city', 'merchant_status']

Payment Methods:
['payment_method_id', 'payment_method_name', 'status']

Devices:
['device_id', 'customer_id', 'device_type', 'operating_system', 'device_status']

Locations:
['location_id', 'city', 'state', 'country']


In [124]:
def generate_transactions(
    customers_df,
    accounts_df,
    merchants_df,
    payment_methods_df,
    devices_df,
    locations_df
):
    """
    Generate synthetic financial transactions
    using existing customers, accounts, merchants,
    payment methods, devices, and locations.
    """

    # --------------------------------------------------------
    # 1. Generate transaction IDs
    # --------------------------------------------------------

    transaction_ids = generate_ids(
        "TXN",
        NUM_TRANSACTIONS
    )

    # --------------------------------------------------------
    # 2. Select accounts for transactions
    # --------------------------------------------------------

    transaction_account_ids = np.random.choice(
        accounts_df["account_id"],
        size=NUM_TRANSACTIONS
    )

    # --------------------------------------------------------
    # 3. Map each account to its customer
    # --------------------------------------------------------

    account_customer_map = dict(
        zip(
            accounts_df["account_id"],
            accounts_df["customer_id"]
        )
    )

    transaction_customer_ids = [
        account_customer_map[account_id]
        for account_id in transaction_account_ids
    ]

    # --------------------------------------------------------
    # 4. Merchant
    # --------------------------------------------------------

    transaction_merchant_ids = np.random.choice(
        merchants_df["merchant_id"],
        size=NUM_TRANSACTIONS
    )

    # --------------------------------------------------------
    # 5. Payment method
    # --------------------------------------------------------

    transaction_payment_method_ids = np.random.choice(
        payment_methods_df["payment_method_id"],
        size=NUM_TRANSACTIONS
    )

    # --------------------------------------------------------
    # 6. Device
    # --------------------------------------------------------

    transaction_device_ids = np.random.choice(
        devices_df["device_id"],
        size=NUM_TRANSACTIONS
    )

    # --------------------------------------------------------
    # 7. Location
    # --------------------------------------------------------

    transaction_location_ids = np.random.choice(
        locations_df["location_id"],
        size=NUM_TRANSACTIONS
    )

    # --------------------------------------------------------
    # 8. Transaction amounts
    # --------------------------------------------------------

    transaction_amounts = np.round(
        np.random.lognormal(
            mean=7.5,
            sigma=1.2,
            size=NUM_TRANSACTIONS
        ),
        2
    )

    # --------------------------------------------------------
    # 9. Transaction types
    # --------------------------------------------------------

    transaction_types = np.random.choice(
        [
            "Purchase",
            "Withdrawal",
            "Transfer",
            "Bill Payment",
            "Refund"
        ],
        size=NUM_TRANSACTIONS,
        p=[0.55, 0.15, 0.15, 0.10, 0.05]
    )

    # --------------------------------------------------------
    # 10. Transaction status
    # --------------------------------------------------------

    transaction_statuses = np.random.choice(
        [
            "Completed",
            "Pending",
            "Failed",
            "Reversed"
        ],
        size=NUM_TRANSACTIONS,
        p=[0.88, 0.05, 0.05, 0.02]
    )

    # --------------------------------------------------------
    # 11. Transaction timestamps
    # --------------------------------------------------------

    start_date = pd.Timestamp("2024-01-01")
    end_date = pd.Timestamp("2026-06-30")

    total_seconds = int(
        (end_date - start_date).total_seconds()
    )

    random_seconds = np.random.randint(
        0,
        total_seconds,
        size=NUM_TRANSACTIONS
    )

    transaction_timestamps = (
        start_date
        + pd.to_timedelta(random_seconds, unit="s")
    )

    # --------------------------------------------------------
    # 12. Create DataFrame
    # --------------------------------------------------------

    transactions_df = pd.DataFrame({
        "transaction_id": transaction_ids,
        "customer_id": transaction_customer_ids,
        "account_id": transaction_account_ids,
        "merchant_id": transaction_merchant_ids,
        "payment_method_id": transaction_payment_method_ids,
        "device_id": transaction_device_ids,
        "location_id": transaction_location_ids,
        "transaction_timestamp": transaction_timestamps,
        "transaction_amount": transaction_amounts,
        "transaction_type": transaction_types,
        "transaction_status": transaction_statuses
    })

    return transactions_df

In [125]:
transactions_df = generate_transactions(
    customers_df,
    accounts_df,
    merchants_df,
    payment_methods_df,
    devices_df,
    locations_df
)

print(
    "Transactions generated:",
    len(transactions_df)
)

transactions_df.head()

Transactions generated: 100000


,transaction_id,customer_id,account_id,merchant_id,payment_method_id,device_id,location_id,transaction_timestamp,transaction_amount,transaction_type,transaction_status
0,TXN000001,CUS002005,ACC006173,MER001478,PM000003,DEV005585,LOC000412,2024-06-03 17:44:07,2133.12,Purchase,Completed
1,TXN000002,CUS000830,ACC013852,MER001534,PM000004,DEV005869,LOC000034,2024-11-25 08:55:09,2752.25,Transfer,Completed
2,TXN000003,CUS005153,ACC001846,MER001486,PM000004,DEV008573,LOC000121,2025-03-30 10:46:00,3484.40,Transfer,Completed
3,TXN000004,CUS003985,ACC012917,MER000147,PM000004,DEV008924,LOC000206,2024-04-10 14:28:42,4247.52,Withdrawal,Completed
4,TXN000005,CUS006545,ACC001347,MER001345,PM000002,DEV000175,LOC000403,2025-07-28 17:50:29,11080.19,Refund,Completed


### **Validate data**

In [126]:
print("========== TRANSACTION VALIDATION ==========")

print("\nNumber of transactions:")
print(len(transactions_df))

print("\nNumber of columns:")
print(len(transactions_df.columns))

print("\nDuplicate Transaction IDs:")
print(
    transactions_df["transaction_id"].duplicated().sum()
)

print("\nMissing Values:")
print(
    transactions_df.isnull().sum()
)

========== TRANSACTION VALIDATION ==========

Number of transactions:
100000

Number of columns:
11

Duplicate Transaction IDs:
0

Missing Values:
transaction_id           0
customer_id              0
account_id               0
merchant_id              0
payment_method_id        0
device_id                0
location_id              0
transaction_timestamp    0
transaction_amount       0
transaction_type         0
transaction_status       0
dtype: int64


## 13:**Fraud Predictions Data Generation**

Fraud predictions capture transaction-level fraud risk assessments generated using behavioral risk signals.

In [127]:
def generate_fraud_predictions(transactions_df):
    """
    Generate synthetic fraud predictions using
    transaction-level behavioral risk signals.
    """

    fraud_prediction_ids = generate_ids(
        "FRD",
        len(transactions_df)
    )

    # --------------------------------------------------------
    # 1. Start with a base risk score
    # --------------------------------------------------------

    fraud_risk_score = np.zeros(
        len(transactions_df)
    )

    # --------------------------------------------------------
    # 2. High transaction amount
    # --------------------------------------------------------

    high_amount_threshold = transactions_df[
        "transaction_amount"
    ].quantile(0.95)

    high_amount = (
        transactions_df["transaction_amount"]
        >= high_amount_threshold
    )

    fraud_risk_score += (
        high_amount.astype(int) * 25
    )

    # --------------------------------------------------------
    # 3. Failed transactions
    # --------------------------------------------------------

    failed_transaction = (
        transactions_df["transaction_status"]
        == "Failed"
    )

    fraud_risk_score += (
        failed_transaction.astype(int) * 15
    )

    # --------------------------------------------------------
    # 4. Reversed transactions
    # --------------------------------------------------------

    reversed_transaction = (
        transactions_df["transaction_status"]
        == "Reversed"
    )

    fraud_risk_score += (
        reversed_transaction.astype(int) * 15
    )

    # --------------------------------------------------------
    # 5. Withdrawal transactions
    # --------------------------------------------------------

    withdrawal = (
        transactions_df["transaction_type"]
        == "Withdrawal"
    )

    fraud_risk_score += (
        withdrawal.astype(int) * 10
    )

    # --------------------------------------------------------
    # 6. Customer transaction behavior
    # --------------------------------------------------------

    customer_average_amount = (
        transactions_df
        .groupby("customer_id")["transaction_amount"]
        .transform("mean")
    )

    unusually_large_transaction = (
        transactions_df["transaction_amount"]
        > customer_average_amount * 3
    )

    fraud_risk_score += (
        unusually_large_transaction.astype(int) * 20
    )

    # --------------------------------------------------------
    # 7. Small random variation
    # --------------------------------------------------------

    fraud_risk_score += np.random.uniform(
        0,
        10,
        size=len(transactions_df)
    )

    # --------------------------------------------------------
    # 8. Keep score between 0 and 100
    # --------------------------------------------------------

    fraud_risk_score = np.clip(
        fraud_risk_score,
        0,
        100
    )

    fraud_risk_score = np.round(
        fraud_risk_score,
        2
    )

    # --------------------------------------------------------
    # 9. Risk level
    # --------------------------------------------------------

    risk_level = pd.cut(
        fraud_risk_score,
        bins=[-1, 30, 60, 100],
        labels=[
            "Low",
            "Medium",
            "High"
        ]
    )

    # --------------------------------------------------------
    # 10. Identify the highest-risk transactions
    # --------------------------------------------------------

    fraud_threshold = fraud_risk_score[
        fraud_risk_score >= 0
    ].mean()

    # Use the top 2% of transactions as fraud cases
    fraud_cutoff = np.percentile(
        fraud_risk_score,
        98
    )

    fraud_prediction = np.where(
        fraud_risk_score >= fraud_cutoff,
        "Fraud",
        "Legitimate"
    )

    # --------------------------------------------------------
    # 11. Prediction timestamp
    # --------------------------------------------------------

    prediction_timestamp = (
        transactions_df["transaction_timestamp"]
        + pd.to_timedelta(
            np.random.randint(
                1,
                300,
                size=len(transactions_df)
            ),
            unit="s"
        )
    )

    # --------------------------------------------------------
    # 12. Create final DataFrame
    # --------------------------------------------------------

    fraud_predictions_df = pd.DataFrame({
        "fraud_prediction_id": fraud_prediction_ids,
        "transaction_id": transactions_df["transaction_id"],
        "risk_score": fraud_risk_score,
        "risk_level": risk_level.astype(str),
        "fraud_prediction": fraud_prediction,
        "prediction_timestamp": prediction_timestamp
    })

    return fraud_predictions_df

In [128]:
fraud_predictions_df = generate_fraud_predictions(
    transactions_df
)

print(
    "Fraud predictions generated:",
    len(fraud_predictions_df)
)

fraud_predictions_df.head()

Fraud predictions generated: 100000


,fraud_prediction_id,transaction_id,risk_score,risk_level,fraud_prediction,prediction_timestamp
0,FRD000001,TXN000001,4.28,Low,Legitimate,2024-06-03 17:47:51
1,FRD000002,TXN000002,9.72,Low,Legitimate,2024-11-25 08:58:33
2,FRD000003,TXN000003,3.36,Low,Legitimate,2025-03-30 10:50:07
3,FRD000004,TXN000004,12.93,Low,Legitimate,2024-04-10 14:28:43
4,FRD000005,TXN000005,25.62,Low,Legitimate,2025-07-28 17:52:01


In [129]:
fraud_predictions_df[
    fraud_predictions_df["fraud_prediction"] == "Fraud"
].head(20)

,fraud_prediction_id,transaction_id,risk_score,risk_level,fraud_prediction,prediction_timestamp
5,FRD000006,TXN000006,51.80,Medium,Fraud,2024-03-29 00:03:25
13,FRD000014,TXN000014,57.99,Medium,Fraud,2024-06-02 10:47:41
102,FRD000103,TXN000103,51.97,Medium,Fraud,2024-05-20 14:35:04
122,FRD000123,TXN000123,52.47,Medium,Fraud,2026-01-25 14:50:19
155,FRD000156,TXN000156,53.86,Medium,Fraud,2026-03-07 22:12:30
166,FRD000167,TXN000167,55.95,Medium,Fraud,2025-08-10 01:30:23
239,FRD000240,TXN000240,54.36,Medium,Fraud,2024-07-09 02:00:09
326,FRD000327,TXN000327,55.41,Medium,Fraud,2025-01-19 08:10:49
367,FRD000368,TXN000368,65.49,High,Fraud,2024-03-24 06:35:14
368,FRD000369,TXN000369,64.78,High,Fraud,2024-08-16 07:45:22


In [130]:
fraud_predictions_df['fraud_prediction'].value_counts()

,count
fraud_prediction,
Legitimate,98000
Fraud,2000


In [131]:
fraud_predictions_df[
    fraud_predictions_df["fraud_prediction"] == "Fraud"
]["risk_score"].describe()

,risk_score
count,2000.000000
mean,56.922130
std,5.449147
min,51.500000
25%,52.940000
50%,54.490000
75%,60.450000
max,79.990000


In [132]:
print(
    fraud_predictions_df["fraud_prediction"]
    .value_counts()
)

print("\nRisk Level:")
print(
    fraud_predictions_df["risk_level"]
    .value_counts()
)

print("\nRisk Score:")
print(
    fraud_predictions_df["risk_score"].describe()
)

fraud_prediction
Legitimate    98000
Fraud          2000
Name: count, dtype: int64

Risk Level:
risk_level
Low       94329
Medium     5132
High        539
Name: count, dtype: int64

Risk Score:
count    100000.000000
mean         10.094473
std          11.293074
min           0.000000
25%           3.420000
50%           6.820000
75%          11.330000
max          79.990000
Name: risk_score, dtype: float64



## 14:**Risk Scores Data Generation**

Risk scores provide customer-level financial risk assessments based on credit and behavioral indicators.

In [133]:
def generate_risk_scores(
    customers_df,
    credit_profiles_df,
    transactions_df,
    support_tickets_df
):
    """
    Generate customer-level synthetic risk scores
    using financial and behavioral indicators.
    """

    # --------------------------------------------------------
    # 1. Customer IDs
    # --------------------------------------------------------

    customer_ids = customers_df["customer_id"].tolist()

    # --------------------------------------------------------
    # 2. Credit information
    # --------------------------------------------------------

    credit_data = credit_profiles_df[
        [
            "customer_id",
            "credit_score",
            "credit_utilization"
        ]
    ].copy()

    # Lower credit score = higher risk
    credit_data["credit_risk"] = (
        100
        - (
            (credit_data["credit_score"] - 300)
            / 600
            * 100
        )
    )

    # Higher utilization = higher risk
    credit_data["utilization_risk"] = (
        credit_data["credit_utilization"]
        .clip(0, 100)
    )

    # --------------------------------------------------------
    # 3. Transaction behavior
    # --------------------------------------------------------

    transaction_summary = (
        transactions_df
        .groupby("customer_id")
        .agg(
            transaction_count=(
                "transaction_id",
                "count"
            ),
            failed_count=(
                "transaction_status",
                lambda x: (x == "Failed").sum()
            ),
            reversed_count=(
                "transaction_status",
                lambda x: (x == "Reversed").sum()
            )
        )
        .reset_index()
    )

    transaction_summary["failed_rate"] = (
        transaction_summary["failed_count"]
        / transaction_summary["transaction_count"]
        * 100
    )

    transaction_summary["reversed_rate"] = (
        transaction_summary["reversed_count"]
        / transaction_summary["transaction_count"]
        * 100
    )

    # --------------------------------------------------------
    # 4. Support ticket activity
    # --------------------------------------------------------

    ticket_summary = (
        support_tickets_df
        .groupby("customer_id")
        .size()
        .reset_index(
            name="support_ticket_count"
        )
    )

    # --------------------------------------------------------
    # 5. Combine customer risk indicators
    # --------------------------------------------------------

    risk_data = (
        pd.DataFrame({
            "customer_id": customer_ids
        })
        .merge(
            credit_data,
            on="customer_id",
            how="left"
        )
        .merge(
            transaction_summary,
            on="customer_id",
            how="left"
        )
        .merge(
            ticket_summary,
            on="customer_id",
            how="left"
        )
    )

    risk_data = risk_data.fillna(0)

    # --------------------------------------------------------
    # 6. Calculate overall risk score
    # --------------------------------------------------------

    risk_data["risk_score"] = (
        risk_data["credit_risk"] * 0.35
        + risk_data["utilization_risk"] * 0.25
        + risk_data["failed_rate"] * 5
        + risk_data["reversed_rate"] * 5
        + risk_data["support_ticket_count"] * 2
    )

    # Keep score between 0 and 100
    risk_data["risk_score"] = np.clip(
        risk_data["risk_score"],
        0,
        100
    )

    risk_data["risk_score"] = np.round(
        risk_data["risk_score"],
        2
    )

    # --------------------------------------------------------
    # 7. Risk category
    # --------------------------------------------------------

    risk_data["risk_category"] = pd.cut(
        risk_data["risk_score"],
        bins=[-1, 30, 60, 100],
        labels=[
            "Low",
            "Medium",
            "High"
        ]
    ).astype(str)

    # --------------------------------------------------------
    # 8. Score timestamp
    # --------------------------------------------------------

    score_timestamp = pd.Timestamp("2026-06-30")

    risk_scores_df = pd.DataFrame({
        "risk_score_id": generate_ids(
            "RSK",
            len(risk_data)
        ),
        "customer_id": risk_data["customer_id"],
        "risk_score": risk_data["risk_score"],
        "risk_category": risk_data["risk_category"],
        "score_timestamp": score_timestamp
    })

    return risk_scores_df

In [134]:
# Generate Risk Scores

risk_scores_df = generate_risk_scores(
    customers_df,
    credit_profiles_df,
    transactions_df,
    support_tickets_df
)

## 15:**Churn Predictions Data Generation**

Churn predictions estimate the likelihood of customers discontinuing their relationship with the institution.

In [135]:
def generate_churn_predictions(
    customers_df,
    transactions_df,
    support_tickets_df,
    risk_scores_df
):
    """
    Generate customer-level synthetic churn predictions
    using customer activity and risk indicators.
    """

    customer_ids = customers_df["customer_id"].tolist()

    # --------------------------------------------------------
    # 1. Transaction behavior
    # --------------------------------------------------------

    transaction_summary = (
        transactions_df
        .groupby("customer_id")
        .agg(
            transaction_count=(
                "transaction_id",
                "count"
            ),
            last_transaction=(
                "transaction_timestamp",
                "max"
            )
        )
        .reset_index()
    )

    # --------------------------------------------------------
    # 2. Calculate recency
    # --------------------------------------------------------

    analysis_date = transactions_df[
        "transaction_timestamp"
    ].max()

    transaction_summary["days_since_transaction"] = (
        analysis_date
        - transaction_summary["last_transaction"]
    ).dt.days

    # --------------------------------------------------------
    # 3. Support activity
    # --------------------------------------------------------

    ticket_summary = (
        support_tickets_df
        .groupby("customer_id")
        .size()
        .reset_index(
            name="support_ticket_count"
        )
    )

    # --------------------------------------------------------
    # 4. Risk score
    # --------------------------------------------------------

    risk_data = risk_scores_df[
        [
            "customer_id",
            "risk_score"
        ]
    ]

    # --------------------------------------------------------
    # 5. Combine everything
    # --------------------------------------------------------

    churn_data = (
        pd.DataFrame({
            "customer_id": customer_ids
        })
        .merge(
            transaction_summary,
            on="customer_id",
            how="left"
        )
        .merge(
            ticket_summary,
            on="customer_id",
            how="left"
        )
        .merge(
            risk_data,
            on="customer_id",
            how="left"
        )
    )

    churn_data = churn_data.fillna(0)

    # --------------------------------------------------------
    # 6. Churn risk score
    # --------------------------------------------------------

    churn_data["churn_score"] = (
        churn_data["days_since_transaction"] * 0.35
        + churn_data["support_ticket_count"] * 5
        + churn_data["risk_score"] * 0.20
    )

    # Normalize into 0-100
    churn_min = churn_data["churn_score"].min()
    churn_max = churn_data["churn_score"].max()

    if churn_max > churn_min:
        churn_data["churn_score"] = (
            (
                churn_data["churn_score"]
                - churn_min
            )
            / (churn_max - churn_min)
            * 100
        )
    else:
        churn_data["churn_score"] = 0

    churn_data["churn_score"] = np.round(
        churn_data["churn_score"],
        2
    )

    # --------------------------------------------------------
    # 7. Churn risk level
    # --------------------------------------------------------

    churn_data["churn_risk_level"] = pd.cut(
        churn_data["churn_score"],
        bins=[-1, 30, 60, 100],
        labels=[
            "Low",
            "Medium",
            "High"
        ]
    ).astype(str)

    # --------------------------------------------------------
    # 8. Churn prediction
    # --------------------------------------------------------

    churn_data["churn_prediction"] = np.where(
        churn_data["churn_score"] >= 60,
        "Likely to Churn",
        "Likely to Stay"
    )

    # --------------------------------------------------------
    # 9. Prediction timestamp
    # --------------------------------------------------------

    prediction_timestamp = pd.Timestamp(
        "2026-06-30"
    )

    churn_predictions_df = pd.DataFrame({
        "churn_prediction_id": generate_ids(
            "CHN",
            len(churn_data)
        ),
        "customer_id": churn_data["customer_id"],
        "churn_score": churn_data["churn_score"],
        "churn_risk_level": churn_data["churn_risk_level"],
        "churn_prediction": churn_data["churn_prediction"],
        "prediction_timestamp": prediction_timestamp
    })

    return churn_predictions_df

In [137]:
# Generate Churn Predictions

churn_predictions_df = generate_churn_predictions(
    customers_df,
    transactions_df,
    support_tickets_df,
    risk_scores_df
)

## 16:**Fraud Investigations Data Generation**

Fraud investigations track the review and resolution process for transactions flagged as potentially fraudulent.


In [136]:
def generate_fraud_investigations(
    fraud_predictions_df
):
    """
    Generate synthetic fraud investigations
    for transactions predicted as fraudulent.
    """

    # --------------------------------------------------------
    # 1. Select fraudulent transactions
    # --------------------------------------------------------

    fraud_cases = fraud_predictions_df[
        fraud_predictions_df["fraud_prediction"] == "Fraud"
    ].copy()

    fraud_cases = fraud_cases.reset_index(
        drop=True
    )

    investigation_count = len(fraud_cases)

    # --------------------------------------------------------
    # 2. Investigation status
    # --------------------------------------------------------

    investigation_status = np.random.choice(
        [
            "Open",
            "Under Review",
            "Resolved",
            "Closed"
        ],
        size=investigation_count,
        p=[0.15, 0.25, 0.35, 0.25]
    )

    # --------------------------------------------------------
    # 3. Investigation priority
    # --------------------------------------------------------

    investigation_priority = np.select(
        [
            fraud_cases["risk_score"] >= 85,
            fraud_cases["risk_score"] >= 70
        ],
        [
            "Critical",
            "High"
        ],
        default="Medium"
    )

    # --------------------------------------------------------
    # 4. Create investigation table
    # --------------------------------------------------------

    fraud_investigations_df = pd.DataFrame({
        "investigation_id": generate_ids(
            "INV",
            investigation_count
        ),
        "transaction_id": fraud_cases[
            "transaction_id"
        ],
        "fraud_prediction_id": fraud_cases[
            "fraud_prediction_id"
        ],
        "investigation_status": investigation_status,
        "investigation_priority": investigation_priority
    })

    return fraud_investigations_df

In [138]:
# Generate Fraud Investigations

fraud_investigations_df = generate_fraud_investigations(
    fraud_predictions_df
)

print("Risk Scores:", len(risk_scores_df))
print("Churn Predictions:", len(churn_predictions_df))
print("Fraud Investigations:", len(fraud_investigations_df))

Risk Scores: 10000
Churn Predictions: 10000
Fraud Investigations: 2000


In [146]:
fraud_investigations_df.head()

,investigation_id,transaction_id,fraud_prediction_id,investigation_status,investigation_priority
0,INV000001,TXN000006,FRD000006,Under Review,Medium
1,INV000002,TXN000014,FRD000014,Closed,Medium
2,INV000003,TXN000103,FRD000103,Resolved,Medium
3,INV000004,TXN000123,FRD000123,Under Review,Medium
4,INV000005,TXN000156,FRD000156,Under Review,Medium


Row counts for all 16 tables

In [139]:
table_counts = {
    "Institution": len(institutions_df),
    "Customer": len(customers_df),
    "Account": len(accounts_df),
    "Card": len(cards_df),
    "Credit Profile": len(credit_profiles_df),
    "Loan": len(loans_df),
    "Merchant": len(merchants_df),
    "Payment Method": len(payment_methods_df),
    "Device": len(devices_df),
    "Location": len(locations_df),
    "Support Ticket": len(support_tickets_df),
    "Transactions": len(transactions_df),
    "Fraud Prediction": len(fraud_predictions_df),
    "Risk Score": len(risk_scores_df),
    "Churn Prediction": len(churn_predictions_df),
    "Fraud Investigation": len(fraud_investigations_df)
}

for table, count in table_counts.items():
    print(f"{table:<25} {count:,}")

Institution               5
Customer                  10,000
Account                   15,000
Card                      12,000
Credit Profile            10,000
Loan                      4,000
Merchant                  2,000
Payment Method            6
Device                    12,000
Location                  500
Support Ticket            5,000
Transactions              100,000
Fraud Prediction          100,000
Risk Score                10,000
Churn Prediction          10,000
Fraud Investigation       2,000


Check duplicate primary IDs

In [140]:
primary_keys = {
    "Institution": ("institution_id", institutions_df),
    "Customer": ("customer_id", customers_df),
    "Account": ("account_id", accounts_df),
    "Card": ("card_id", cards_df),
    "Credit Profile": ("credit_profile_id", credit_profiles_df),
    "Loan": ("loan_id", loans_df),
    "Merchant": ("merchant_id", merchants_df),
    "Payment Method": ("payment_method_id", payment_methods_df),
    "Device": ("device_id", devices_df),
    "Location": ("location_id", locations_df),
    "Support Ticket": ("ticket_id", support_tickets_df),
    "Transactions": ("transaction_id", transactions_df),
    "Fraud Prediction": ("fraud_prediction_id", fraud_predictions_df),
    "Risk Score": ("risk_score_id", risk_scores_df),
    "Churn Prediction": ("churn_prediction_id", churn_predictions_df),
    "Fraud Investigation": ("investigation_id", fraud_investigations_df)
}

for table, (column, df) in primary_keys.items():
    duplicates = df[column].duplicated().sum()
    print(f"{table:<25} duplicates: {duplicates}")

Institution               duplicates: 0
Customer                  duplicates: 0
Account                   duplicates: 0
Card                      duplicates: 0
Credit Profile            duplicates: 0
Loan                      duplicates: 0
Merchant                  duplicates: 0
Payment Method            duplicates: 0
Device                    duplicates: 0
Location                  duplicates: 0
Support Ticket            duplicates: 0
Transactions              duplicates: 0
Fraud Prediction          duplicates: 0
Risk Score                duplicates: 0
Churn Prediction          duplicates: 0
Fraud Investigation       duplicates: 0


## **Export all 16 CSVs**

In [141]:
import os

# Create data folder
os.makedirs("data", exist_ok=True)

# Export all 16 tables
institutions_df.to_csv("data/institution.csv", index=False)
customers_df.to_csv("data/customer.csv", index=False)
accounts_df.to_csv("data/account.csv", index=False)
cards_df.to_csv("data/card.csv", index=False)
credit_profiles_df.to_csv("data/credit_profile.csv", index=False)
loans_df.to_csv("data/loan.csv", index=False)
merchants_df.to_csv("data/merchant.csv", index=False)
payment_methods_df.to_csv("data/payment_method.csv", index=False)
devices_df.to_csv("data/device.csv", index=False)
locations_df.to_csv("data/location.csv", index=False)
support_tickets_df.to_csv("data/support_ticket.csv", index=False)
transactions_df.to_csv("data/transactions.csv", index=False)
fraud_predictions_df.to_csv("data/fraud_prediction.csv", index=False)
risk_scores_df.to_csv("data/risk_score.csv", index=False)
churn_predictions_df.to_csv("data/churn_prediction.csv", index=False)
fraud_investigations_df.to_csv("data/fraud_investigation.csv", index=False)

print("All 16 tables exported successfully.")

All 16 tables exported successfully.


In [144]:
import shutil

shutil.make_archive(
    "sufinex_data",
    "zip",
    "data"
)

print("Created sufinex_data.zip")

Created sufinex_data.zip


In [145]:
from google.colab import files

files.download("sufinex_data.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>